# Progressive B Model Sweep

This notebook launches `scripts/train_progressive_b_lm.py` with external corpus options, model-size sweeps, JSON/CSV artifacts, and TensorBoard logging.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src").exists() and (path / "scripts").exists()
)
print({"project_root": str(project_root), "python": sys.executable, "matplotlib": plt is not None})

In [ ]:
RUN_NAME = "notebook_sweep"
TEXT_FILE = "artifacts/data/sample.txt"
TEXT_SOURCES: list[str] = []
JSONL_SOURCES: list[str] = []
JSONL_TEXT_KEYS = ["text"]
HF_DATASET = ""
HF_CONFIG = ""
HF_SPLIT = "train"
HF_TEXT_KEY = "text"
HF_STREAMING = False
CORPUS_MAX_SAMPLES: int | None = None
CORPUS_MAX_CHARS: int | None = 20000

TRAINING_OBJECTIVE = "teacher_forcing"
TEACHER_FORCING_CHUNK_SIZE: int | None = 8
TOKENIZER = "char"
SUBWORD_VOCAB_SIZE = 256
SEQ_LEN = 32
BATCH_SIZE = 8
STEPS = 50
EVAL_INTERVAL = 10
EVAL_STEPS = 2
SAMPLE_TOKENS = 60
PROMPT_TEXT = "Jakal-Net explores "
SWEEP_PRESETS = ["tiny", "small", "base"]
OUTPUT_DIR = "artifacts/training_runs"
USE_TENSORBOARD = True

In [ ]:
command = [
    sys.executable,
    str(project_root / "scripts" / "train_progressive_b_lm.py"),
    "--run-name", RUN_NAME,
    "--output-dir", OUTPUT_DIR,
    "--training-objective", TRAINING_OBJECTIVE,
    "--tokenizer", TOKENIZER,
    "--subword-vocab-size", str(SUBWORD_VOCAB_SIZE),
    "--seq-len", str(SEQ_LEN),
    "--batch-size", str(BATCH_SIZE),
    "--steps", str(STEPS),
    "--eval-interval", str(EVAL_INTERVAL),
    "--eval-steps", str(EVAL_STEPS),
    "--sample-tokens", str(SAMPLE_TOKENS),
    "--prompt-text", PROMPT_TEXT,
    "--sweep-presets", ",".join(SWEEP_PRESETS),
]
if TEXT_FILE:
    command.extend(["--text-file", TEXT_FILE])
for source in TEXT_SOURCES:
    command.extend(["--text-source", source])
for source in JSONL_SOURCES:
    command.extend(["--jsonl-source", source])
for key in JSONL_TEXT_KEYS:
    command.extend(["--jsonl-text-key", key])
if HF_DATASET:
    command.extend(["--hf-dataset", HF_DATASET, "--hf-split", HF_SPLIT, "--hf-text-key", HF_TEXT_KEY])
    if HF_CONFIG:
        command.extend(["--hf-config", HF_CONFIG])
    if HF_STREAMING:
        command.append("--hf-streaming")
if CORPUS_MAX_SAMPLES is not None:
    command.extend(["--corpus-max-samples", str(CORPUS_MAX_SAMPLES)])
if CORPUS_MAX_CHARS is not None:
    command.extend(["--corpus-max-chars", str(CORPUS_MAX_CHARS)])
if TRAINING_OBJECTIVE == "teacher_forcing" and TEACHER_FORCING_CHUNK_SIZE is not None:
    command.extend(["--teacher-forcing-chunk-size", str(TEACHER_FORCING_CHUNK_SIZE)])
if USE_TENSORBOARD:
    command.append("--tensorboard")

result = subprocess.run(
    command,
    cwd=project_root,
    text=True,
    capture_output=True,
    check=True,
)
print(result.stdout)
session_dir_line = next(line for line in result.stdout.splitlines() if line.startswith("session_dir="))
SESSION_DIR = Path(session_dir_line.split("=", 1)[1].strip())
print({"session_dir": str(SESSION_DIR)})

In [ ]:
summaries = json.loads((SESSION_DIR / "session_summary.json").read_text(encoding="utf-8"))
for row in summaries:
    print({
        "experiment": row["experiment"],
        "params": row["parameter_count"],
        "best_val_loss": round(row["best_val_loss"], 4),
        "best_val_ppl": round(row["best_val_ppl"], 2),
        "runtime_seconds": round(row["runtime_seconds"], 2),
        "tensorboard_dir": row["tensorboard_dir"],
    })

if plt is not None and summaries:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    labels = [row["experiment"] for row in summaries]
    axes[0].bar(labels, [row["best_val_loss"] for row in summaries])
    axes[0].set_title("Best Val Loss")
    axes[0].set_ylabel("loss")
    axes[1].bar(labels, [row["parameter_count"] for row in summaries])
    axes[1].set_title("Parameter Count")
    axes[1].set_ylabel("params")
    plt.tight_layout()
    plt.show()

In [ ]:
best_run = min(summaries, key=lambda row: row["best_val_loss"])
best_dir = Path(best_run["run_dir"])
print({"best_experiment": best_run["experiment"], "best_run_dir": str(best_dir)})
print("\n--- generated sample ---\n")
print((best_dir / "sample_generated.txt").read_text(encoding="utf-8"))

if best_run.get("tensorboard_dir"):
    print("\nTensorBoard:")
    print(f"{sys.executable} -m tensorboard.main --logdir {best_run['tensorboard_dir']}")